# Autoregressive rollout evaluation

rollout条件とGTデータを確認してから自己回帰推論を実行し、誤差成長と保存済み画像・ParaView出力を可視化します。YAMLはNotebookから変更しません。


## 1. 使用する設定


In [ ]:
from pathlib import Path
import csv, subprocess, sys
import matplotlib.pyplot as plt
import yaml
from IPython.display import Image, Markdown, display

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "configs").is_dir() and (candidate / "performance").is_dir():
            return candidate
    raise FileNotFoundError("FNOV2リポジトリ内からNotebookを開いてください。")

def resolve(value, config_path):
    path = Path(value)
    return path.resolve() if path.is_absolute() else (config_path.parent / path).resolve()

def status(ok, label, detail=""):
    mark = "✅" if ok else "❌"
    print(f"{mark} {label}" + (f": {detail}" if detail else ""))
    return ok

ROOT = find_project_root(Path.cwd())
print("project:", ROOT)
################################
EXPERIMENT_ID = "ex004_baseline"
################################
print("experiment:", EXPERIMENT_ID)
ROLLOUT_YAML = ROOT / "performance/configs/rollout.yaml"
cfg=yaml.safe_load(ROLLOUT_YAML.read_text(encoding="utf-8"))
loader_path=resolve(cfg["test_dataloader_config"],ROLLOUT_YAML)
loader=yaml.safe_load(loader_path.read_text(encoding="utf-8"))
kind=loader["evaluation"]["type"]
data_dir=resolve(loader["data"]["directory"],loader_path)
checkpoint=resolve(cfg["checkpoint"],ROLLOUT_YAML)
result_dir=resolve(cfg["output"]["root_directory"],ROLLOUT_YAML)/kind/"rollout"


## 2. 設定の可視化


In [ ]:
summary={
 "evaluation":kind,"data directory":str(data_dir),"loader test Re":loader["test_re"],
 "rollout Re":cfg["rollout"]["reynolds"],"start/end":[cfg["rollout"]["start_time"],cfg["rollout"]["end_time"]],
 "steps":cfg["rollout"]["steps"],"checkpoint":str(checkpoint),"W&B":cfg["wandb"],"output":str(result_dir),
}
print(yaml.safe_dump(summary,allow_unicode=True,sort_keys=False))
display(Markdown("### rollout.yaml")); print(ROLLOUT_YAML.read_text(encoding="utf-8"))
display(Markdown("### test_dataloader.yaml")); print(loader_path.read_text(encoding="utf-8"))


## 3. 実行前チェック


In [ ]:
files=sorted(data_dir.glob(loader["data"]["file_pattern"]))
available=set()
for path in files:
    try: available.add(float(path.stem.rsplit("_Re",1)[1]))
    except (IndexError,ValueError): pass
requested={float(v) for v in cfg["rollout"]["reynolds"]}
loader_re={float(v) for v in loader["test_re"]}
performance_root=(ROOT/"performance/data"/kind).resolve()
r=cfg["rollout"]
expected_end=float(r["start_time"])+int(r["steps"])*(float(r["end_time"])-float(r["start_time"]))/int(r["steps"])
checks=[
 status(ROLLOUT_YAML.is_file(),"rollout config",ROLLOUT_YAML),
 status(loader_path.is_file(),"test dataloader config",loader_path),
 status(checkpoint.is_file(),"checkpoint",checkpoint),
 status(data_dir.is_dir(),"performance data directory",data_dir),
 status(data_dir.is_relative_to(performance_root),"data isolation",performance_root),
 status(requested.issubset(loader_re),"rollout Re in test_re",f"requested={sorted(requested)}"),
 status(requested.issubset(available),"rollout Re files",f"available={sorted(available)}"),
 status(abs(expected_end-float(r["end_time"]))<=float(r["tolerance"]),"time/step consistency"),
 status(result_dir.parents[1].name==EXPERIMENT_ID,"experiment result root",result_dir.parents[1]),
]
PRECHECK_OK=all(checks)
print("\n実行可能" if PRECHECK_OK else "\n設定またはデータを修正してください。")


## 4. Rollout実行


In [ ]:
RUN_ROLLOUT = True
if RUN_ROLLOUT:
    if not PRECHECK_OK: raise RuntimeError("事前チェックが失敗しています。")
    subprocess.run([sys.executable,str(ROOT/"performance/src/rollout.py"),"--config",str(ROLLOUT_YAML)],cwd=ROOT,check=True)
else: print("未実行。確認後にRUN_ROLLOUT=Trueへ変更してください。")


## 5. Re別の誤差成長


In [ ]:
metrics_path=result_dir/cfg["output"]["metrics_csv"]
if not metrics_path.exists(): print("結果がありません:",metrics_path)
else:
 with metrics_path.open(encoding="utf-8",newline="") as f: rows=list(csv.DictReader(f))
 grouped={}
 for row in rows: grouped.setdefault(float(row["Re"]),[]).append(row)
 fig,axes=plt.subplots(1,2,figsize=(13,4.5))
 for re_value,values in sorted(grouped.items()):
  t=[float(v["time"]) for v in values]
  axes[0].plot(t,[float(v["relative_l2_overall"]) for v in values],label=f"Re {re_value:g}")
  axes[1].plot(t,[float(v["mse_overall"]) for v in values],label=f"Re {re_value:g}")
 axes[0].set(xlabel="time [s]",ylabel="Relative L2",title="Rollout error growth")
 axes[1].set(xlabel="time [s]",ylabel="MSE",title="Rollout MSE")
 for ax in axes: ax.grid(alpha=.3); ax.legend()
 plt.tight_layout(); plt.show()


## 6. GT / Prediction / Absolute Error画像

表示するReを変更してセルを再実行できます。既にPNGとして保存された画像をNotebook内へ表示します。


In [ ]:
SELECTED_RE=cfg["rollout"]["reynolds"][4]
plot_dir=result_dir/"plots"/f"Re{SELECTED_RE:g}"
images=sorted(plot_dir.glob("*.png"))
if not images: print("画像がありません:",plot_dir)
else:
 print(f"Re {SELECTED_RE:g}: {len(images)} images")
 for path in images:
  display(Markdown(f"### {path.stem}"))
  display(Image(filename=str(path)))


## 7. PT・ParaView出力の確認


In [ ]:
for re_value in cfg["rollout"]["reynolds"]:
 status((result_dir/f"rollout_Re{re_value:g}.pt").exists(),f"full PT Re {re_value:g}")
 status((result_dir/f"rollout_prediction_Re{re_value:g}.pt").exists(),f"prediction PT Re {re_value:g}")
 pvd=result_dir/"vti"/f"Re{re_value:g}"/f"prediction_Re{re_value:g}.pvd"
 status(pvd.exists(),f"ParaView PVD Re {re_value:g}",pvd)
